# Library Installation

In [11]:
# Install all required libraries in one command
# !pip install tensorflow numpy pandas scikit-learn matplotlib scipy optuna seaborn joblib plotly nbformat>=4.2.0 -q

# Preprocessing

In [12]:
import pandas as pd
import warnings
import os

warnings.filterwarnings('ignore')

df_non_null = pd.read_csv('/root/vynixmodelling/dataset/mixed_df.csv')
df_triple_barrier_result = pd.read_csv('/root/vynixmodelling/ML_RL/triple_barrier_non_null.csv')

In [13]:
df_mix = pd.concat([df_non_null.iloc[-len(df_triple_barrier_result):].reset_index(drop=True), df_triple_barrier_result.reset_index(drop=True)], axis=1)

In [14]:
pd.DataFrame.to_csv(df_mix, "df_mix.csv")

# Train-Test Split

In [15]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

def create_train_test_split_by_date(df_mix, test_start_year=2022):
    """
    Split data menjadi train dan test berdasarkan tahun.
    Test data dimulai dari tahun yang ditentukan.
    
    Parameters:
    -----------
    df_mix: pandas.DataFrame
        DataFrame yang berisi data gabungan
    test_start_year: int
        Tahun mulai test data
        
    Returns:
    --------
    train_data: pandas.DataFrame
        Data training
    test_data: pandas.DataFrame
        Data testing
    """
    df_mix['date'] = pd.to_datetime(df_mix['date'])
    
    # Split berdasarkan tahun
    train_data = df_mix[df_mix['date'].dt.year < test_start_year]
    test_data = df_mix[df_mix['date'].dt.year >= test_start_year]
    
    return train_data, test_data

# Gunakan fungsi untuk split data
train_data, test_data = create_train_test_split_by_date(df_mix, test_start_year=2022)

# Import Libraries for GMM-HMM

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Import untuk HMM
from hmmlearn import hmm

# Set random seed for reproducibility
np.random.seed(42)

# GMM-HMM Model Implementation

In [17]:
class StockGMMHMM:
    """
    GMM-HMM untuk analisis saham dengan Gaussian Mixture Model - Hidden Markov Model.
    
    Model ini menggabungkan:
    - Hidden Markov Model (HMM) untuk memodelkan state tersembunyi
    - Gaussian Mixture Model (GMM) untuk distribusi observasi di setiap state
    
    Attributes:
    -----------
    n_states: int
        Jumlah state tersembunyi
    n_mix: int
        Jumlah komponen Gaussian mixture per state
    covariance_type: str
        Tipe matriks kovarians ('full', 'diag', 'spherical')
    random_state: int
        Seed untuk reproducibility
    model: hmm.GMMHMM
        Model GMM-HMM yang telah dilatih
    """
    
    def __init__(self, n_states=3, n_mix=2, covariance_type='full', random_state=42):
        """
        Inisialisasi model GMM-HMM.
        
        Parameters:
        -----------
        n_states: int, default=3
            Jumlah state tersembunyi
        n_mix: int, default=2
            Jumlah komponen Gaussian mixture
        covariance_type: str, default='full'
            Tipe matriks kovarians
        random_state: int, default=42
            Seed untuk reproducibility
        """
        self.n_states = n_states
        self.n_mix = n_mix
        self.covariance_type = covariance_type
        self.random_state = random_state
        self.model = None
    
    def preprocess_data(self, df, selected_features):
        """
        Preprocess data untuk model GMM-HMM.
        
        Parameters:
        -----------
        df: pandas.DataFrame
            DataFrame berisi data saham
        selected_features: list
            List nama kolom fitur yang akan digunakan
            
        Returns:
        --------
        X_scaled: numpy.ndarray
            Data yang telah dipreprocess dan diskalakan
        """
        # Pilih fitur yang ditentukan
        X = df[selected_features].values
        
        # Handle missing values
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Normalisasi data
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Tambahkan noise kecil untuk stabilitas numerik
        np.random.seed(self.random_state)
        X_scaled += np.random.normal(0, 1e-6, X_scaled.shape)
        
        return X_scaled
    
    def train(self, X):
        """
        Melatih model GMM-HMM.
        
        Parameters:
        -----------
        X: numpy.ndarray
            Data terpreprocessing untuk pelatihan
            
        Returns:
        --------
        self: StockGMMHMM
            Instance model yang telah dilatih
        """
        # Inisialisasi model GMM-HMM dengan parameter yang lebih stabil
        self.model = hmm.GMMHMM(
            n_components=self.n_states,
            n_mix=self.n_mix,
            covariance_type=self.covariance_type,
            random_state=self.random_state,
            n_iter=100,
            tol=0.01,
            init_params='kmeans',
            params='stmcw'
        )
        
        try:
            # Coba latih model
            self.model.fit(X)
        except Exception as e:
            print(f"Error dalam pelatihan: {str(e)}")
            
            # Jika gagal, coba dengan parameter yang lebih stabil
            self.covariance_type = 'diag'
            self.model = hmm.GMMHMM(
                n_components=self.n_states,
                n_mix=self.n_mix,
                covariance_type='diag',
                random_state=self.random_state,
                n_iter=100,
                tol=0.1,
                init_params='kmeans',
                params='stmcw'
            )
            
            # Tambahkan regularisasi ke matriks kovarians
            self.model.covars_prior = 0.1
            self.model.fit(X)
        
        return self
    
    def predict_states(self, X):
        """
        Memprediksi state tersembunyi untuk data input.
        
        Parameters:
        -----------
        X: numpy.ndarray
            Data terpreprocessing untuk prediksi
            
        Returns:
        --------
        states: numpy.ndarray
            State tersembunyi yang diprediksi
        state_probs: numpy.ndarray
            Probabilitas state
        """
        try:
            # Prediksi state
            states = self.model.predict(X)
            
            # Hitung probabilitas state
            logprob, state_probs = self.model.score_samples(X)
            
            return states, state_probs
        except Exception as e:
            print(f"Error dalam prediksi: {str(e)}")
            
            # Alternatif: Hitung probabilitas secara manual
            log_probs = np.zeros((len(X), self.n_states))
            
            # Hitung log probabilitas untuk setiap state
            for i in range(self.n_states):
                # Untuk setiap mixture, hitung probabilitas
                for j in range(self.n_mix):
                    # Hitung mahalanobis distance
                    means = self.model.means_[i, j]
                    
                    if self.covariance_type == 'diag':
                        covars = self.model.covars_[i, j]
                        precision = 1.0 / covars
                        log_det = np.sum(np.log(covars))
                    else:
                        # Untuk tipe kovarians lain, gunakan implementasi sederhana
                        precision = np.eye(X.shape[1])
                        log_det = 0
                    
                    # Hitung log probabilitas untuk mixture ini
                    for k in range(len(X)):
                        diff = X[k] - means
                        log_probs[k, i] += self.model.weights_[i, j] * np.exp(
                            -0.5 * np.sum(diff**2 * precision) - 0.5 * log_det
                        )
            
            # Normalisasi dan temukan state dengan probabilitas tertinggi
            log_probs = log_probs / np.sum(log_probs, axis=1, keepdims=True)
            states = np.argmax(log_probs, axis=1)
            
            return states, log_probs

# Helper Functions

In [18]:
def calculate_confusion_matrix(states, labels):
    """
    Menghitung matriks M yang menghitung frekuensi label terhadap state.
    
    Parameters:
    -----------
    states: numpy.ndarray
        State tersembunyi yang diprediksi model
    labels: numpy.ndarray
        Label sebenarnya (-1, 0, 1)
        
    Returns:
    --------
    M: numpy.ndarray
        Matriks penghitungan M[i,j]
    """
    # Jumlah state unik
    n_states = len(np.unique(states))
    
    # Inisialisasi matriks M
    M = np.zeros((n_states, 3))
    
    # Mapping label ke indeks
    label_map = {-1: 0, 0: 1, 1: 2}
    
    # Hitung frekuensi
    for i in range(len(states)):
        state = int(states[i])
        label_idx = label_map.get(labels[i], 1)  # Default ke netral jika tidak diketahui
        M[state, label_idx] += 1
    
    return M

def calculate_MR_matrix(M):
    """
    Menghitung matriks rasio penghitungan MR dari matriks M.
    
    Parameters:
    -----------
    M: numpy.ndarray
        Matriks penghitungan
        
    Returns:
    --------
    MR: numpy.ndarray
        Matriks rasio penghitungan
    """
    # Jumlah per baris
    row_sums = M.sum(axis=1, keepdims=True)
    
    # Hindari pembagian dengan nol
    row_sums = np.where(row_sums == 0, 1, row_sums)
    
    # Hitung MR
    MR = M / row_sums
    
    return MR

def calculate_accuracy(MR):
    """
    Menghitung akurasi per state berdasarkan matriks MR.
    
    Parameters:
    -----------
    MR: numpy.ndarray
        Matriks rasio penghitungan
        
    Returns:
    --------
    Acc: numpy.ndarray
        Akurasi per state
    """
    # Akurasi per state adalah nilai maksimum di setiap baris MR
    Acc = np.max(MR, axis=1)
    
    return Acc

def calculate_entropy(MR):
    """
    Menghitung entropi per state berdasarkan matriks MR.
    
    Parameters:
    -----------
    MR: numpy.ndarray
        Matriks rasio penghitungan
        
    Returns:
    --------
    H: numpy.ndarray
        Entropi per state
    """
    # Hindari log(0)
    epsilon = 1e-10
    
    # Hitung entropi
    H = -np.sum(MR * np.log(MR + epsilon), axis=1)
    
    return H

def calculate_weights(M):
    """
    Menghitung bobot per state.
    
    Parameters:
    -----------
    M: numpy.ndarray
        Matriks penghitungan
        
    Returns:
    --------
    w: numpy.ndarray
        Bobot per state
    """
    # Jumlah per state
    state_sums = np.sum(M, axis=1)
    
    # Total keseluruhan
    total_sum = np.sum(M)
    
    # Bobot
    w = state_sums / total_sum
    
    return w

def calculate_model_score(Acc, H, w):
    """
    Menghitung skor model berdasarkan akurasi, entropi, dan bobot.
    
    Parameters:
    -----------
    Acc: numpy.ndarray
        Akurasi per state
    H: numpy.ndarray
        Entropi per state
    w: numpy.ndarray
        Bobot per state
        
    Returns:
    --------
    score: float
        Skor model keseluruhan
    """
    # Hitung skor sesuai formula
    score = np.sum(Acc * (1 / (1 + H)) * w)
    
    return score

# Model Training and Evaluation

In [19]:
def train_and_evaluate_gmm_hmm(df_mix, n_states=3, n_mix=2):
    """
    Melatih dan mengevaluasi model GMM-HMM menggunakan dataset yang sudah digabung.
    
    Parameters:
    -----------
    df_mix: pandas.DataFrame
        DataFrame berisi data saham yang sudah digabung dengan hasil triple barrier
    n_states: int
        Jumlah state tersembunyi
    n_mix: int
        Jumlah komponen Gaussian mixture
        
    Returns:
    --------
    results: dict
        Dictionary berisi hasil evaluasi
    """
    # Inisialisasi model dengan parameter yang lebih stabil
    model = StockGMMHMM(n_states=n_states, n_mix=n_mix, covariance_type='diag')
    
    # Tentukan fitur yang akan digunakan
    selected_features = [
        'close', 'Volume',
        'MACD', 'Signal',
        'K', 'D'
    ]
    
    try:
        # Preprocess data
        X = model.preprocess_data(df_mix, selected_features)
        
        # Latih model
        model.train(X)
        
        # Prediksi state
        states, state_probs = model.predict_states(X)
        
        # Hasil dasar
        results = {
            'model': model,
            'states': states,
            'state_probs': state_probs
        }
        
        # Evaluasi akurasi model menggunakan label dari df_mix
        if 'label' in df_mix.columns:
            try:
                # Ambil label dari df_mix
                labels = df_mix['label'].values
                
                # Hitung matriks M
                M = calculate_confusion_matrix(states, labels)
                
                # Hitung matriks MR
                MR = calculate_MR_matrix(M)
                
                # Hitung akurasi per state
                Acc = calculate_accuracy(MR)
                
                # Hitung entropi per state
                H = calculate_entropy(MR)
                
                # Hitung bobot per state
                w = calculate_weights(M)
                
                # Hitung skor model
                score = calculate_model_score(Acc, H, w)
                
                # Tambahkan hasil evaluasi ke dictionary
                results.update({
                    'M': M,
                    'MR': MR,
                    'Acc': Acc,
                    'H': H,
                    'w': w,
                    'score': score
                })
            except Exception as e:
                print(f"Warning: Tidak dapat mengevaluasi akurasi model: {str(e)}")
        
        return results
    
    except Exception as e:
        print(f"Error dalam pelatihan dan evaluasi: {str(e)}")
        return None

# Run Model Training

In [20]:
# Contoh penggunaan model GMM-HMM dengan dataset yang sudah digabung
results = train_and_evaluate_gmm_hmm(df_mix, n_states=3, n_mix=2)